# Random Forest

Adapted from PR #4: shared stratified 80/10/10 CSVs, with the kids group excluded. Previous outputs were cleared because the data preparation changed. Run parameter comparisons on validation data before running the final test cell.


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score


features = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms']

from pathlib import Path

data_dir = Path('data') if Path('data/train.csv').exists() else Path('../data')
train = pd.read_csv(data_dir / 'train.csv')
val = pd.read_csv(data_dir / 'validation.csv')
test = pd.read_csv(data_dir / 'test.csv')

X_train, y_train = train[features], train['genre_group']
X_val, y_val = val[features], val['genre_group']
X_test, y_test = test[features], test['genre_group']

print(len(train), len(val), len(test))  # should be roughly 80 / 10 / 10

### using balanced_subsample for class weight

In [ ]:
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight='balanced_subsample', random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_val)
print(classification_report(y_val, rf_pred))

### using unbalanced for class weight

In [ ]:
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight=None, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_val)
print(classification_report(y_val, rf_pred))

# testing/tweaking parameters

#### changing min_samples_leaf and max_features

Unbalanced ----------

In [ ]:
best_score = 0
best_params = None

for leaf in [1, 3, 5, 10]:
    for mf in ['sqrt', 0.5, 1.0]:
        rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf, max_features=mf, n_jobs=-1, random_state=42)
        rf.fit(X_train, y_train)
        score = f1_score(y_val, rf.predict(X_val), average='macro')
        print(f"min_samples_leaf={leaf:<3} max_features={mf!s:<5} val macro-F1 = {score:.3f}")

        if score > best_score:
            best_score = score
            best_params = {'min_samples_leaf': leaf, 'max_features': mf}

print("\nBest:", best_params, f"-> {best_score:.3f}")

Balanced ----------

In [ ]:
best_score = 0
best_params = None

for leaf in [1, 3, 5, 10]:
    for mf in ['sqrt', 0.5, 1.0]:
        rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf, max_features=mf, class_weight='balanced_subsample', n_jobs=-1, random_state=42)
        rf.fit(X_train, y_train)
        score = f1_score(y_val, rf.predict(X_val), average='macro')
        print(f"min_samples_leaf={leaf:<3} max_features={mf!s:<5} val macro-F1 = {score:.3f}")

        if score > best_score:
            best_score = score
            best_params = {'min_samples_leaf': leaf, 'max_features': mf}

print("\nBest:", best_params, f"-> {best_score:.3f}")

In [ ]:
best_score = 0
best_est = 0

for est in [100, 200, 300, 400, 500]:
    rf = RandomForestClassifier(n_estimators=est, min_samples_leaf=best_params['min_samples_leaf'], max_features=best_params['max_features'], class_weight='balanced_subsample', n_jobs=-1, random_state=42)
    rf.fit(X_train, y_train)
    score = f1_score(y_val, rf.predict(X_val), average='macro')
    print(f"n_estimators={est:<3} val macro-F1 = {score:.3f}")

    if score > best_score:
        best_score = score
        best_est = est

print("\nBest:", best_est, f"-> {best_score:.3f}")

# Results

In [ ]:
rf = RandomForestClassifier(n_estimators=500, min_samples_leaf=3, max_features=0.5, class_weight='balanced_subsample',  n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_val)
print(classification_report(y_val, rf_pred))

In [ ]:
print(classification_report(y_test, rf.predict(X_test)))